# Pancancer enrichment analysis step 3: Make figures

## Setup

In [1]:
import cptac
import cptac.utils as ut
import os
import datetime
import altair as alt
import numpy as np
import pandas as pd

In [2]:
TIME_START = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

STEP2_DIR = "step2_outputs_reactome"
STEP2_FILE = "urls_20200609_122817_from_tumor_change_20200529_195104_10000_perm.tsv"
STEP2_FILE_PATH = os.path.join(STEP2_DIR, STEP2_FILE)

## Read in data from step 2

In [3]:
data = pd.read_csv(STEP2_FILE_PATH, sep="\t", index_col=0)

In [4]:
data.head()

,pathway_id,cancer_type,mean_expr,url,name,pathway_times_enriched,pathway_avg_fdr,pathway_avg_rank,pathway_avg_overlap,cancer_type_rank,entities_ratio,entities_pValue,entities_fdr,entities_found,entities_total
0,R-HSA-6798695,ccrcc,0.233503,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.0,0.033072,0,0.033072,3.467008e-03,8.426404e-01,448,480
1,R-HSA-6798695,colon,-0.134831,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.0,0.033072,0,0.033072,6.216916e-12,1.500142e-08,434,480
2,R-HSA-6798695,endometrial,0.288515,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.0,0.033072,0,0.033072,1.403496e-03,8.219205e-01,450,480
3,R-HSA-6798695,gbm,0.166227,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.0,0.033072,0,0.033072,2.129677e-03,8.037977e-01,448,480
4,R-HSA-6798695,hnscc,0.133159,https://reactome.org/PathwayBrowser/#/R-HSA-67...,Neutrophil degranulation,8,0.587721,0.0,0.033072,0,0.033072,3.649260e-03,8.329209e-01,462,480


## Figure 1: Bubble plot

In [39]:
data = data.assign(rank_size=-1 * np.log10(data["cancer_type_rank"] + 1))
mean_expr_max = max(data["mean_expr"])
mean_expr_min = min(data["mean_expr"])

alt.Chart(data).mark_circle().encode(
    x=alt.X(
        "name:N",
        sort=data["name"].values,
        axis=alt.Axis(
            labelAngle=20,
            labelFontSize=12,
            labelLimit=500,
            title="<--- Overall more enriched | Overall less enriched --->",
            titleFontSize=12
        )
    ),
    y=alt.Y(
        "cancer_type:N",
        axis=alt.Axis(
            title="Cancer type",
            titleFontSize=12
        ),
    ),
    size="rank_size:Q",
    color=alt.Color(
        "mean_expr:Q",
        scale=alt.Scale(
            scheme="redblue",
            domain=[mean_expr_max, mean_expr_min]
        )
    )
).properties(
    width=600,
    height=400
).configure_axis(
    grid=True
)

alt.Chart(...)

In [ ]:
def barplot_most_enriched(data, cancer_type):
    
    data = data[data["cancer_type"] == cancer_type].\
        sort_values(by=["entities_fdr", "entities_pValue"]).\
        iloc[0:NUM_TOP, :]
    
    plot = alt.Chart(data).mark_bar().encode(
        x=alt.X(
            "name:N",
            sort=data["name"].values,
            axis=alt.Axis(
                title="<---- Lower FDR | Higher FDR ---->",
                labelAngle=20,
                labelFontSize=14,
                labelLimit=500,
                titleFontSize=16
            )
        ),
        y=alt.Y(
            "expr_mean:Q",
            axis=alt.Axis(
                title="Pathway mean expression change",
                titleFontSize=16
            )
        ),
        color=alt.condition(
            alt.datum.expr_mean > 0,
            alt.value("#fde500"),
            alt.value("#04429b")
        )
    ).properties(
        width=600,
        height=500,
        title=f"{cancer_type.title()} top enriched pathways"
    ).configure_title(
        fontSize=16
    )
    
    plot.save(f"/home/caleb/Downloads/{cancer_type}.png")

    return plot